# Geneformer Self-SAE Training

Train a sparse autoencoder on native Geneformer pair embeddings using the same self-reconstruction objective as the SLformer SAE. This provides a fair reconstruction-quality control before comparing SLformer and Geneformer manifold distance ordering.


In [ ]:
from __future__ import annotations

import pickle as pkl
import sys
import warnings
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", message="IProgress not found*")
warnings.filterwarnings("ignore", message=r"The epoch parameter in `scheduler\.step\(\)` was not necessary*")

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "SAE").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from SAE.SAE_training.model import SAEConfig, SparseAutoencoder
from SAE.SAE_training.utils.diagnostics import corr_offdiag, effective_rank, summ
from SAE.SAE_training.utils.plotting import load_metrics, plot_training_summary
from SAE.SAE_training.utils.training import fit_and_save_sae, load_sae_train_config, setup_experiment

sns.set_theme(style="white", context="paper")
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 140})

print("Project root:", PROJECT_ROOT)
print("Using src dir:", SRC_DIR)
print("Torch:", torch.__version__, "CUDA available:", torch.cuda.is_available())


## Configuration

The Geneformer self-SAE uses the same training configuration as the SLformer SAE, except the input dimension is the native Geneformer pair dimension $d=512$ and W&B logging is disabled for this local control run. The output directory is separate from SLformer checkpoints.


In [ ]:
CONFIG_PATH = PROJECT_ROOT / "src" / "SAE" / "SAE_training" / "config" / "train_config.yaml"
cfg, slformer_model_cfg, train_cfg = load_sae_train_config(CONFIG_PATH)
paths_cfg = cfg["paths"]
scope_cfg = cfg["scope"]
norm_cfg = cfg["normalization"]

GENEFORMER_PREDICTION_CSVS = [PROJECT_ROOT / "data" / "all_SL" / f"pred_mix_geneformer_nokg_cv{fold}.csv" for fold in range(1, 6)]
GENEFORMER_EMB_PKL = PROJECT_ROOT / "data" / "saved_data" / "map" / "geneformer_emb.pkl"
GENE2ID_PKL = PROJECT_ROOT / "data" / "saved_data" / "map" / "gene2id.pkl"
CANCER_LIST_TXT = PROJECT_ROOT / "data" / "saved_data" / "map" / "cancer_list.txt"
OUTPUT_BASE_DIR = Path(paths_cfg["output_base_dir"]).expanduser().resolve()

CANCER = str(scope_cfg["cancer"])
SEED = int(scope_cfg["seed"])
VAL_SPLIT = float(cfg["train"]["val_split"])
EPS = float(norm_cfg["eps"])

model_cfg = replace(slformer_model_cfg, d_in=512)
run_name = f"hidden{model_cfg.d_hidden}_gate{model_cfg.gate_weight}_orth{model_cfg.orth_weight}_k{model_cfg.topk}"
run_dir = OUTPUT_BASE_DIR / "geneformer_self" / CANCER / run_name
run_dir.mkdir(parents=True, exist_ok=True)

setup_experiment(seed=train_cfg.seed)

print("Config:", CONFIG_PATH)
print("Prediction CSVs:", GENEFORMER_PREDICTION_CSVS[0], "...", GENEFORMER_PREDICTION_CSVS[-1])
print("Geneformer embeddings:", GENEFORMER_EMB_PKL)
print("Output run dir:", run_dir)
print("Model config:", model_cfg)
print("Train config:", train_cfg)


## Build Native Geneformer Pair Matrix

For each row $i=(g_i^A,g_i^B,c_i)$, the native Geneformer pair vector is

$$
G_i=[h^{\mathrm{GF}}_{g_i^A,c_i}; h^{\mathrm{GF}}_{g_i^B,c_i}]\in\mathbb{R}^{512}.
$$

This notebook trains the SAE on $G$ directly, rather than projecting Geneformer into SLformer space.


In [ ]:
geneformer_prediction_parts = [pd.read_csv(path) for path in GENEFORMER_PREDICTION_CSVS]
meta = pd.concat(geneformer_prediction_parts, ignore_index=True).reset_index(drop=True)
meta["fold"] = np.repeat(np.arange(len(geneformer_prediction_parts)), [len(part) for part in geneformer_prediction_parts])

with GENE2ID_PKL.open("rb") as f:
    gene2id_map = pkl.load(f)
with CANCER_LIST_TXT.open() as f:
    cancer_list = [line.strip() for line in f if line.strip()]
cancer2id_map = {cancer: index for index, cancer in enumerate(cancer_list)}
with GENEFORMER_EMB_PKL.open("rb") as f:
    geneformer_emb = pkl.load(f)

G_geneformer = np.stack([
    np.concatenate([
        geneformer_emb[cancer2id_map[str(row.cancer)]][gene2id_map[str(row.primary_gene)]],
        geneformer_emb[cancer2id_map[str(row.cancer)]][gene2id_map[str(row.partner_gene)]],
    ])
    for row in meta.itertuples(index=False)
]).astype(np.float32)

y = meta["score"].to_numpy(dtype=np.float32)

print("G_geneformer:", G_geneformer.shape, G_geneformer.dtype)
print("Score range:", float(y.min()), "->", float(y.max()))
print("Cancer counts:", meta["cancer"].value_counts().sort_index().to_dict())
display(meta.head())


## Train/Validation Split and Normalization

The normalization is fit only on the training rows:

$$
\tilde G_{ij}=\frac{G_{ij}-\mu_j}{\sigma_j+\varepsilon}.
$$

The validation split matches the SLformer training protocol by using the same seed and validation fraction.


In [ ]:
all_idx = np.arange(len(meta))
train_idx, val_idx = train_test_split(all_idx, test_size=VAL_SPLIT, random_state=SEED, shuffle=True)
G_train = G_geneformer[train_idx]
G_val = G_geneformer[val_idx]

mu = G_train.mean(axis=0, dtype=np.float64)
sigma = G_train.std(axis=0, dtype=np.float64) + EPS
G_train_n = ((G_train - mu) / sigma).astype(np.float32)
G_val_n = ((G_val - mu) / sigma).astype(np.float32)

norm_dir = run_dir / "norm"
norm_dir.mkdir(parents=True, exist_ok=True)
np.save(norm_dir / "mu.npy", mu.astype(np.float32))
np.save(norm_dir / "sigma.npy", sigma.astype(np.float32))

print("Train:", G_train_n.shape, "Val:", G_val_n.shape)
print("Saved norm into:", norm_dir)


## Train Geneformer Self-SAE

The optimization problem is the same reconstruction objective used for SLformer SAE training:

$$
\min_\theta\; \|D_\theta f_\theta(\tilde G_i)-\tilde G_i\|_2^2
+\lambda_g\|\gamma\|_1+\lambda_{orth}\Omega(D_\theta).
$$

This makes reconstruction quality comparable as a self-reconstruction diagnostic for each native model space.


In [ ]:
model, best_ckpt, log_dir = fit_and_save_sae(
    G_train_n,
    sae_cfg=model_cfg,
    train_cfg=train_cfg,
    out_dir=run_dir,
    X_val=G_val_n,
    wandb_run=None,
)

print("Best checkpoint:", best_ckpt)
print("Log dir:", log_dir)

if best_ckpt is not None and Path(best_ckpt).exists():
    ckpt = torch.load(best_ckpt, map_location="cpu", weights_only=False)
    torch.save({"sae_cfg": ckpt["sae_cfg"], "state_dict": ckpt["state_dict"]}, run_dir / "final.pt")
else:
    torch.save({"sae_cfg": model_cfg.__dict__, "state_dict": model.state_dict()}, run_dir / "final.pt")
print("Saved:", run_dir / "final.pt")

metrics_df = load_metrics(run_dir / "metrics.csv")
print("Metrics rows:", len(metrics_df))
display(metrics_df.tail())


## Training Diagnostics

The first figure shows whether the self-SAE reached a stable validation reconstruction loss without collapse in latent support.


In [ ]:
metrics_df = load_metrics(run_dir / "metrics.csv")
plot_training_summary(metrics_df, title=f"Geneformer self-SAE training: loss + latent metrics ({CANCER})", show_val=True)


## Reconstruction Error and Geometry Preservation

The held-out reconstruction is summarized by standardized MSE/MAE and by whether sample-sample correlation structure and effective rank are preserved. These diagnostics should be interpreted as reconstruction quality, not SL clinical evidence.


In [ ]:
ckpt = torch.load(run_dir / "final.pt", map_location="cpu", weights_only=False)
sae_cfg_loaded = SAEConfig(**ckpt["sae_cfg"])
model_diag = SparseAutoencoder(sae_cfg_loaded)
model_diag.load_state_dict(ckpt["state_dict"])
model_diag.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_diag.to(device)

with torch.no_grad():
    G_val_hat_t, z_val_t = model_diag(torch.from_numpy(G_val_n).to(device))
G_val_hat = G_val_hat_t.detach().cpu().numpy().astype(np.float32)
z_val = z_val_t.detach().cpu().numpy().astype(np.float32)

recon_summary = pd.DataFrame([
    {
        "split": "validation",
        "standardized_recon_mse": mean_squared_error(G_val_n, G_val_hat),
        "standardized_recon_mae": mean_absolute_error(G_val_n, G_val_hat),
        "latent_l0_mean": float((np.abs(z_val) > 1e-6).sum(axis=1).mean()),
        "latent_dead_frac": float(((np.abs(z_val) > 1e-6).sum(axis=0) == 0).mean()),
    }
])
display(recon_summary)

c0 = corr_offdiag(G_val_n)
c1 = corr_offdiag(G_val_hat)
print("Sample-sample corr off-diag summary (orig):", summ(c0))
print("Sample-sample corr off-diag summary (recon):", summ(c1))

fig, ax = plt.subplots(figsize=(6.2, 3.5), constrained_layout=True)
ax.hist(c0, bins=60, alpha=0.55, label="orig", density=True)
ax.hist(c1, bins=60, alpha=0.55, label="recon", density=True)
ax.set_xlabel("pairwise correlation (off-diagonal)")
ax.set_ylabel("density")
ax.set_title("Geneformer self-SAE correlation structure")
ax.legend(frameon=False)
plt.show()

A0 = G_val_n - G_val_n.mean(axis=1, keepdims=True)
A1 = G_val_hat - G_val_hat.mean(axis=1, keepdims=True)
S0 = np.linalg.svd(A0, full_matrices=False, compute_uv=False)
S1 = np.linalg.svd(A1, full_matrices=False, compute_uv=False)
print("Effective rank (orig, recon):", effective_rank(S0), effective_rank(S1))
print("Top-1 variance fraction (orig, recon):", float((S0[0] ** 2) / (np.sum(S0**2) + 1e-12)), float((S1[0] ** 2) / (np.sum(S1**2) + 1e-12)))
